# FieldLens ML 01: Colab setup and dataset check

**Goal of this notebook (Day 11):** prove the Colab environment works and that the
dataset split we built locally arrives intact. Real training starts on Day 12.

Steps:
1. Check the GPU
2. Load `processed.zip` from Google Drive
3. Verify the split matches `docs/dataset.md`
4. Build the data pipeline
5. Compute class weights
6. Build the model skeleton (not trained yet)

Dataset: Kaggle "Garbage Classification" (ODbL). Split made by
`apps/ml/scripts/make_split.py` (seed 42, group-aware, 70/15/15).

## Step 1: Is the GPU on?

In Colab: **Runtime > Change runtime type > T4 GPU**, then re-run this cell.

In [ ]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
assert gpus, "No GPU found. Runtime > Change runtime type > T4 GPU"
!nvidia-smi

## Step 2: Get the dataset

On your computer, zip the **processed** folder (not raw) and upload it to Google Drive as
`MyDrive/fieldlens/processed.zip`. Then run this cell.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

ZIP_PATH = "/content/drive/MyDrive/fieldlens/processed.zip"
!rm -rf /content/data && mkdir -p /content/data
!unzip -q "$ZIP_PATH" -d /content/data
!ls /content/data

## Step 3: Verify the split

The zip may contain `processed/` directly or one extra folder level, so we search for it.
The counts must match the table in `docs/dataset.md` exactly. If they do not, stop and
find out why before training on it.

In [ ]:
from pathlib import Path
import pandas as pd

root = next(Path("/content/data").rglob("manifest.csv")).parent
print("Dataset root:", root)

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png"}
SPLITS = ["train", "val", "test"]
EXPECTED = {  # from docs/dataset.md
    "GLASS":   (1407, 302, 301),
    "METAL":   (538, 115, 115),
    "ORGANIC": (689, 148, 148),
    "PAPER":   (1359, 291, 291),
    "PLASTIC": (605, 130, 129),
}

rows = []
for cls, expected in EXPECTED.items():
    counts = tuple(
        sum(1 for f in (root / s / cls).iterdir() if f.suffix.lower() in IMAGE_SUFFIXES)
        for s in SPLITS
    )
    rows.append({"class": cls, **dict(zip(SPLITS, counts)), "ok": counts == expected})

table = pd.DataFrame(rows)
display(table)
assert table["ok"].all(), "Counts differ from docs/dataset.md"

manifest = pd.read_csv(root / "manifest.csv")
print("Manifest rows:", len(manifest), "| groups:", manifest["group_id"].nunique())

# No duplicate group may sit in more than one split.
leaks = (manifest.groupby("group_id")["split"].nunique() > 1).sum()
print("Groups spanning several splits:", leaks)
assert leaks == 0

## Step 4: Data pipeline

Images are resized to 224x224. The seed is fixed. Only the train set is shuffled.
Augmentation is **not** added here. On Day 12 it goes inside the model and applies only
during training, so val and test stay untouched.

In [ ]:
IMG_SIZE = (224, 224)
BATCH = 32
SEED = 42

def load(split, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        root / split,
        image_size=IMG_SIZE,
        batch_size=BATCH,
        shuffle=shuffle,
        seed=SEED,
        label_mode="int",
    )

train_ds = load("train", shuffle=True)
val_ds = load("val", shuffle=False)
test_ds = load("test", shuffle=False)

class_names = train_ds.class_names  # alphabetical order
print("Classes:", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

Look at a few training images to make sure labels match what you see.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
for images, labels in train_ds.take(1):
    for i in range(8):
        plt.subplot(2, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i])])
        plt.axis("off")
plt.show()

## Step 5: Class weights

Formula: `weight = total / (n_classes x class_count)`, computed on the **train** split only.
These should match the weights in `docs/dataset.md` (0.65, 1.71, 1.33, 0.68, 1.52).

In [ ]:
train_counts = {c: EXPECTED[c][0] for c in class_names}
total = sum(train_counts.values())
n_classes = len(class_names)

class_weight = {
    i: total / (n_classes * train_counts[name]) for i, name in enumerate(class_names)
}
for i, name in enumerate(class_names):
    print(f"{name:8} train={train_counts[name]:5}  weight={class_weight[i]:.2f}")

## Step 6: Model skeleton (not trained)

A pretrained MobileNetV3 with a new 5-class head. The base is frozen. This is **stage 1**
of transfer learning (train the head). Fine-tuning comes later.

We only build and summarise it today. Training is Day 12.

In [ ]:
base = tf.keras.applications.MobileNetV3Large(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet",
    pooling="avg",
)
base.trainable = False  # frozen: stage 1 trains only the head

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = base(inputs, training=False)  # Keras MobileNetV3 scales 0-255 inputs itself
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(n_classes, activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

## Sanity check

One forward pass through the untrained model. With random head weights the accuracy is
about chance (20% for 5 classes). That is expected. It only proves the pipeline works.

In [ ]:
loss, acc = model.evaluate(val_ds, verbose=0)
print(f"Untrained val accuracy: {acc:.2%} (chance is about 20%)")

## Next (Day 12)

- Add augmentation layers (train only)
- Train the head with `class_weight=class_weight`
- Evaluate per class on val: precision, recall, confusion matrix, macro-F1
- Then fine-tune the last layers at a low learning rate
- Convert to int8 TFLite using a representative dataset drawn from the **train** split